# CFP Ranker '26: Data Collection
This notebook maintains one SQLite database, `espn_cfb_2026.db`, as the **source of truth** for raw ESPN data. Running it again only calls ESPN for data that is missing or may have changed. Team defensive stats are also stored here, derived two ways: by summing player box scores, and by taking each opponent's team box score. Both they are recomputed from the database on every run, with no ESPN requests. Other derived data (rollups, rankings) is computed in later notebooks.

## When each kind of data is fetched

| Data | Fetched when |
| --- | --- |
| FBS teams | The `Teams` table has no FBS teams yet (or `force=True`) |
| Daily scoreboard | The day was never fetched; or it is within the last 7 / next 14 days and was last fetched over 24 hours ago; or it was fetched before that day's games could have finished |
| Game summary (box score + play-by-play) | A Final game has no stored box score or no stored play-by-play; or ESPN had not published one of them yet and the last check was over 24 hours ago; or the game kicked off within the last 7 days and the last check was over 24 hours ago |

Both come from the same ESPN summary request, so one request per game fills both. A re-checked box score or play-by-play is only rewritten if ESPN's data actually changed (compared by hash).

## Tables

| Table | Purpose |
| --- | --- |
| `Teams` | FBS teams plus any FCS opponents seen in games |
| `Games` | Every scoreboard event (any status) |
| `Box_Scores` | One row per game, team and statistic, parsed from the raw payloads |
| `Raw_Boxscores` | The raw ESPN `boxscore` JSON per game (compressed), its hash and when it was fetched. `Box_Scores` can be rebuilt from this with no network calls |
| `Scoreboard_Fetches` | Which days of the scoreboard have been fetched, and when |
| `Player_Box_Scores` | One row per game, team, player, category and statistic, parsed from the raw payloads |
| `Player_Boxscore_Fetches` | The raw payload hash each game's player rows were last parsed from |
| `Raw_Plays` | The raw ESPN `drives` and `scoringPlays` JSON per game (compressed), its hash and when it was fetched. `Plays` and `Scoring_Plays` can be rebuilt from this with no network calls |
| `Plays` | One row per play (columns from `map_schema.flatten_plays`), parsed from `Raw_Plays` |
| `Scoring_Plays` | One row per score: who scored, how many points, and whether the scoring team had the ball when the play started (`is_offense_score`) |
| `Team_Defensive_Stats` | Per game and team, the sum of each `defensive` / `interceptions` player stat. Rebuilt from `Player_Box_Scores` on every run |
| `Team_Defense_Opponent_Stats` | Per game and team, the opponent's `Box_Scores` stats renamed from the defense's point of view (e.g. `totalYardsAllowed`, `takeaways`) plus `pointsAllowed`, and `offensePointsAllowed` from `Scoring_Plays`. Rebuilt from `Box_Scores`, `Games` and `Scoring_Plays` on every run |

**First run on an existing database:** raw payloads were not stored before, so every Final game is fetched once (about 700 requests) to fill `Raw_Boxscores`. After that, runs make few or no requests. The same applies to play-by-play: the first run after it was added fetches every Final game once (about 700 requests) to fill `Raw_Plays`.

## 1. Setup

In [68]:
import hashlib
import json
import sqlite3
import time
import zlib
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from datetime import time as dtime
 
import pandas as pd
import requests

# Play-by-play parsing lives in map_schema.py so the notebook and the schema script share one definition.
from map_schema import collect_drives, flatten_plays
 
# --- Season and database ---
YEAR = 2025
DB_FILE = f"espn_cfb_{YEAR}.db"
SEASON_START = date(YEAR, 8, 1)
SEASON_END = date(YEAR + 1, 2, 1)
 
# --- Refresh rules ---
RECENT_DAYS = 7            # games/days this recent are re-checked for revisions
LOOKAHEAD_DAYS = 14        # upcoming days whose schedules are kept fresh
REFRESH_AFTER_HOURS = 24   # minimum time between re-checks of the same item
REQUEST_DELAY = 0.15       # seconds to wait after each API call
 
# --- ESPN endpoints ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.api.espn.com/apis/site/v2/sports/football/college-football/summary"
STANDINGS_URL = f"https://site.web.api.espn.com/apis/v2/sports/football/college-football/standings?season={YEAR}"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json",
}

_session = requests.Session()
_session.headers.update(HEADERS)
api_calls = 0  # every network request goes through api_get, so this counts them all

def utcnow():
    return datetime.now(timezone.utc)
 
def iso(dt):
    return dt.isoformat(timespec="seconds")

def parse_espn_date(value):
    """Parse ESPN timestamps like '2026-08-29T16:00Z' into timezone-aware datetimes."""
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    return ts.to_pydatetime()

def api_get(url, params=None, retries=3):
    """GET JSON from ESPN with simple retries. Raises requests.RequestException if all attempts fail."""
    global api_calls
    last_exc = None
    for attempt in range(retries):
        api_calls += 1
        try:
            response = _session.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            time.sleep(REQUEST_DELAY)
            return data
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(1.5 * (attempt + 1))
    raise last_exc

## 2. Database

In [69]:
conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON")
 
conn.executescript("""
CREATE TABLE IF NOT EXISTS Teams (
    team_id TEXT PRIMARY KEY,
    display_name TEXT NOT NULL,
    short_name TEXT,
    conference TEXT,
    conference_id TEXT
);
CREATE TABLE IF NOT EXISTS Games (
    game_id TEXT PRIMARY KEY,
    game_date TEXT,
    name TEXT,
    week INTEGER,
    home_team_id TEXT REFERENCES Teams(team_id),
    away_team_id TEXT REFERENCES Teams(team_id),
    home_score INTEGER,
    away_score INTEGER,
    status TEXT,
    venue TEXT,
    neutral_site INTEGER
);
CREATE TABLE IF NOT EXISTS Box_Scores (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, stat_name)
);
CREATE TABLE IF NOT EXISTS Player_Box_Scores (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),
    player_id TEXT NOT NULL,
    player_name TEXT,
    stat_category TEXT NOT NULL,
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, player_id, stat_category, stat_name)
);
CREATE INDEX IF NOT EXISTS idx_player_box_scores_team
    ON Player_Box_Scores(team_id);
CREATE INDEX IF NOT EXISTS idx_player_box_scores_category
    ON Player_Box_Scores(stat_category, stat_name);
-- Raw ESPN "boxscore" section per game. payload is zlib-compressed JSON.
-- payload/payload_hash are NULL when ESPN had not published a box score at fetch time.
CREATE TABLE IF NOT EXISTS Raw_Boxscores (
    game_id TEXT PRIMARY KEY REFERENCES Games(game_id),
    fetched_at TEXT NOT NULL,
    payload_hash TEXT,
    payload BLOB
);
CREATE TABLE IF NOT EXISTS Scoreboard_Fetches (
    game_date TEXT PRIMARY KEY,   -- YYYY-MM-DD
    fetched_at TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Player_Boxscore_Fetches (
    game_id TEXT PRIMARY KEY REFERENCES Games(game_id),
    payload_hash TEXT
);
-- Raw ESPN "drives" + "scoringPlays" sections per game. payload is zlib-compressed JSON.
-- payload/payload_hash are NULL when ESPN had not published play-by-play at fetch time.
CREATE TABLE IF NOT EXISTS Raw_Plays (
    game_id TEXT PRIMARY KEY REFERENCES Games(game_id),
    fetched_at TEXT NOT NULL,
    payload_hash TEXT,
    payload BLOB
);
-- One row per play, parsed from Raw_Plays (columns match map_schema.flatten_plays).
CREATE TABLE IF NOT EXISTS Plays (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    drive_number INTEGER NOT NULL,
    drive_id TEXT,
    drive_team_id TEXT,
    drive_team_name TEXT,
    drive_result TEXT,
    drive_description TEXT,
    play_number_in_drive INTEGER NOT NULL,
    play_id TEXT,
    sequence_number TEXT,
    period INTEGER,
    clock TEXT,
    play_type_id TEXT,
    play_type_text TEXT,
    play_type_abbreviation TEXT,
    text TEXT,
    start_down INTEGER,
    start_distance INTEGER,
    start_yard_line INTEGER,
    start_yards_to_endzone INTEGER,
    start_team_id TEXT,
    end_down INTEGER,
    end_distance INTEGER,
    end_yard_line INTEGER,
    end_yards_to_endzone INTEGER,
    end_team_id TEXT,
    stat_yardage INTEGER,
    home_score INTEGER,
    away_score INTEGER,
    scoring_play INTEGER,
    is_penalty INTEGER,
    is_turnover INTEGER,
    wallclock TEXT,
    modified TEXT,
    PRIMARY KEY (game_id, drive_number, play_number_in_drive)
);
CREATE INDEX IF NOT EXISTS idx_plays_play_id ON Plays(game_id, play_id);
-- One row per score, parsed from Raw_Plays. is_offense_score = 1 when the scoring team had
-- the ball at the start of the play (an offensive score), 0 for defensive/return scores
-- and safeties, NULL when the scoring play could not be matched to a play.
CREATE TABLE IF NOT EXISTS Scoring_Plays (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    score_number INTEGER NOT NULL,
    play_id TEXT,
    period INTEGER,
    clock TEXT,
    team_id TEXT,
    points INTEGER,
    play_type_text TEXT,
    scoring_type TEXT,
    possession_team_id TEXT,
    is_offense_score INTEGER,
    text TEXT,
    PRIMARY KEY (game_id, score_number)
);
-- Derived: team totals of player defensive stats. Fully rebuilt each run.
CREATE TABLE IF NOT EXISTS Team_Defensive_Stats (
    game_id TEXT NOT NULL,
    team_id TEXT NOT NULL,
    stat_category TEXT NOT NULL,
    stat_name TEXT NOT NULL,
    team_total REAL,
    PRIMARY KEY (game_id, team_id, stat_category, stat_name)
);
-- Derived: each team's defense, described by its opponent's team box score. Fully rebuilt each run.
CREATE TABLE IF NOT EXISTS Team_Defense_Opponent_Stats (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),      -- the defending team
    opponent_id TEXT NOT NULL REFERENCES Teams(team_id),  -- the offense it faced
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, stat_name)
);
CREATE INDEX IF NOT EXISTS idx_box_scores_team ON Box_Scores(team_id);
CREATE INDEX IF NOT EXISTS idx_games_date ON Games(game_date);
""")
conn.commit()
print(f"SQLite database ready: {DB_FILE}")

SQLite database ready: espn_cfb_2025.db


## 3. Teams

Fetched once. FCS opponents are added automatically when they appear in the schedule.

In [70]:
def parse_standings_node(node, parent_conference="", parent_conf_id=""):
    """Recursively walk ESPN's standings tree and return teams with their conference name and ID."""
    teams = []
    current_name = node.get("name", "")
    current_id = str(node.get("id", ""))
 
    # Top-level league labels must not override the real conference name/ID
    is_root_label = current_name.lower() in ["fbs", "ncaa - football bowl subdivision", "football bowl subdivision"]
    if is_root_label:
        conf_label, conf_id = "", ""
    else:
        conf_label = parent_conference if parent_conference else current_name
        conf_id = parent_conf_id if parent_conf_id else current_id
 
    for entry in node.get("standings", {}).get("entries", []):
        team = entry.get("team", {})
        if team:
            teams.append({
                "team_id": team.get("id"),
                "display_name": team.get("displayName"),
                "short_name": team.get("shortDisplayName"),
                "conference": conf_label,
                "conference_id": conf_id,
            })
 
    for child in node.get("children", []):
        teams.extend(parse_standings_node(child, parent_conference=conf_label, parent_conf_id=conf_id))
    return teams

def ensure_team(team_id, display_name):
    """Make sure a team row exists (needed for foreign keys). Never overwrites an existing row."""
    if team_id is not None:
        conn.execute(
            "INSERT OR IGNORE INTO Teams (team_id, display_name) VALUES (?, ?)",
            (str(team_id), display_name or "Unknown"),
        )

def load_fbs_teams(force=False):
    stored = conn.execute(
        "SELECT COUNT(*) FROM Teams WHERE conference IS NOT NULL AND conference != ''"
    ).fetchone()[0]
    if stored and not force:
        print(f"Teams: {stored} FBS teams already stored, no fetch needed.")
        return
 
    data = api_get(STANDINGS_URL, params={"group": "80"})  # 80 = FBS
    teams = {t["team_id"]: t for t in parse_standings_node(data) if t["team_id"]}
    conn.executemany(
        """INSERT INTO Teams (team_id, display_name, short_name, conference, conference_id)
           VALUES (?, ?, ?, ?, ?)
           ON CONFLICT(team_id) DO UPDATE SET
               display_name = excluded.display_name, short_name = excluded.short_name,
               conference = excluded.conference, conference_id = excluded.conference_id""",
        [(str(t["team_id"]), t["display_name"], t["short_name"],
          t["conference"] or None, t["conference_id"] or None) for t in teams.values()],
    )
    # Teams stored as FBS from an earlier fetch but not FBS in this season (e.g. 2026 additions) lose their conference.
    conn.execute(
        f"UPDATE Teams SET conference = NULL, conference_id = NULL WHERE team_id NOT IN ({','.join('?' * len(teams))})",
        [str(t) for t in teams],
    )
    conn.commit()
    print(f"Teams: loaded {len(teams)} FBS teams.")

## 4. Schedule (Daily Scoreboard)

In [71]:
GAME_COLUMNS = ["game_id", "game_date", "name", "week", "home_team_id", "away_team_id",
                "home_score", "away_score", "status", "venue", "neutral_site"]
 
GAME_UPSERT = """
INSERT INTO Games (game_id, game_date, name, week, home_team_id, away_team_id,
                   home_score, away_score, status, venue, neutral_site)
VALUES (:game_id, :game_date, :name, :week, :home_team_id, :away_team_id,
        :home_score, :away_score, :status, :venue, :neutral_site)
ON CONFLICT(game_id) DO UPDATE SET
    game_date = excluded.game_date, name = excluded.name, week = excluded.week,
    home_team_id = excluded.home_team_id, away_team_id = excluded.away_team_id,
    home_score = excluded.home_score, away_score = excluded.away_score,
    status = excluded.status, venue = excluded.venue, neutral_site = excluded.neutral_site
"""

def int_or_none(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

def parse_event(event):
    competition = (event.get("competitions") or [{}])[0]
    competitors = competition.get("competitors", [])
    home = next((c for c in competitors if c.get("homeAway") == "home"), {})
    away = next((c for c in competitors if c.get("homeAway") == "away"), {})
    home_team, away_team = home.get("team", {}), away.get("team", {})
    neutral = competition.get("neutralSite")
    return {
        "game_id": str(event.get("id")),
        "game_date": event.get("date"),
        "name": event.get("name"),
        "week": int_or_none((event.get("week") or {}).get("number")),
        "home_team_id": str(home_team["id"]) if home_team.get("id") is not None else None,
        "home_team": home_team.get("displayName"),
        "away_team_id": str(away_team["id"]) if away_team.get("id") is not None else None,
        "away_team": away_team.get("displayName"),
        "home_score": int_or_none(home.get("score")),
        "away_score": int_or_none(away.get("score")),
        "status": ((event.get("status") or {}).get("type") or {}).get("description"),
        "venue": (competition.get("venue") or {}).get("fullName"),
        "neutral_site": None if neutral is None else int(bool(neutral)),
    }

def scoreboard_dates_to_fetch():
    """Days whose scoreboard is missing or may have changed."""
    now = utcnow()
    today = now.date()
    last_day = min(SEASON_END, today + timedelta(days=LOOKAHEAD_DAYS))
    recent_start = today - timedelta(days=RECENT_DAYS)
    stale_before = now - timedelta(hours=REFRESH_AFTER_HOURS)
    fetched = {d: datetime.fromisoformat(t)
               for d, t in conn.execute("SELECT game_date, fetched_at FROM Scoreboard_Fetches")}
 
    dates, day = [], SEASON_START
    while day <= last_day:
        seen = fetched.get(day.isoformat())
        if seen is None:
            dates.append(day)  # never fetched
        else:
            # Late games end by about noon UTC the next day; anything fetched before that may be incomplete
            games_over_by = datetime.combine(day + timedelta(days=1), dtime(12, 0), tzinfo=timezone.utc)
            if seen < stale_before and (day >= recent_start or seen < games_over_by):
                dates.append(day)
        day += timedelta(days=1)
    return dates

def refresh_schedule():
    stats = Counter()
    for day in scoreboard_dates_to_fetch():
        try:
            data = api_get(SCOREBOARD_URL, params={
                "dates": day.strftime("%Y%m%d"), "groups": "80", "limit": "500"})
        except requests.RequestException:
            stats["failed_days"] += 1  # not recorded, so it is retried next run
            continue
 
        for event in data.get("events", []):
            game = parse_event(event)
            ensure_team(game["home_team_id"], game["home_team"])
            ensure_team(game["away_team_id"], game["away_team"])
 
            old = conn.execute(
                "SELECT " + ", ".join(GAME_COLUMNS[1:]) + " FROM Games WHERE game_id = ?",
                (game["game_id"],)).fetchone()
            new = tuple(game[c] for c in GAME_COLUMNS[1:])
            if old is None:
                stats["new_games"] += 1
            elif old != new:
                stats["changed_games"] += 1
            conn.execute(GAME_UPSERT, game)
 
        conn.execute(
            """INSERT INTO Scoreboard_Fetches (game_date, fetched_at) VALUES (?, ?)
               ON CONFLICT(game_date) DO UPDATE SET fetched_at = excluded.fetched_at""",
            (day.isoformat(), iso(utcnow())))
        conn.commit()
        stats["days_fetched"] += 1
 
    print("Schedule:", ", ".join(f"{k}={stats[k]}" for k in
          ["days_fetched", "new_games", "changed_games", "failed_days"]))
    return stats

## 5. Box Scores and Play-by-Play Fetching
ESPN's `boxscore` section is stored raw (compressed) in `Raw_Boxscores`, and parsed into `Box_Scores`. Parsing is a separate step, so a parsing fix only needs `rebuild_box_scores_from_raw()` and no new requests.

The same summary response also carries the play-by-play (`drives` and `scoringPlays`), so `ingest_box_scores` stores that too (section 7). A game is fetched when either part is missing or due for a re-check.

`stat_value` uses ESPN's numeric `value` when it has one, and otherwise falls back to a numeric `displayValue`. Compound values such as `5-12` (made-attempts) or `31:22` stay `NULL` in `stat_value`; the original text is always in `display_value`.

In [72]:
BOXSCORE_INSERT = """
INSERT INTO Box_Scores (game_id, team_id, stat_name, stat_label, stat_value, display_value)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(game_id, team_id, stat_name) DO UPDATE SET
    stat_label = excluded.stat_label, stat_value = excluded.stat_value,
    display_value = excluded.display_value
"""

RAW_UPSERT = """
INSERT INTO Raw_Boxscores (game_id, fetched_at, payload_hash, payload) VALUES (?, ?, ?, ?)
ON CONFLICT(game_id) DO UPDATE SET
    fetched_at = excluded.fetched_at, payload_hash = excluded.payload_hash, payload = excluded.payload
"""

def pack_boxscore(boxscore):
    """Return (hash, compressed bytes) for a boxscore dict. The hash is of a canonical JSON form."""
    canonical = json.dumps(boxscore, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest(), zlib.compress(canonical, 9)

def unpack_boxscore(blob):
    return json.loads(zlib.decompress(blob))

def numeric_or_none(value):
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def parse_team_stats(game_id, boxscore):
    """Flatten a boxscore dict into Box_Scores rows."""
    rows = []
    for team in boxscore.get("teams", []):
        team_id = team.get("team", {}).get("id")
        if team_id is None:
            continue
        for stat in team.get("statistics", []):
            stat_name = stat.get("name") or stat.get("label")
            if not stat_name:
                continue
            value = numeric_or_none(stat.get("value"))
            if value is None:
                value = numeric_or_none(stat.get("displayValue"))
            rows.append((game_id, str(team_id), str(stat_name), stat.get("label"),
                         value, stat.get("displayValue")))
    return rows

def store_boxscore(game_id, boxscore, payload_hash, blob, fetched_at):
    """Write the raw payload and its parsed rows together (the caller commits)."""
    for team in boxscore.get("teams", []):
        info = team.get("team", {})
        ensure_team(info.get("id"), info.get("displayName"))
    conn.execute("DELETE FROM Box_Scores WHERE game_id = ?", (game_id,))
    conn.executemany(BOXSCORE_INSERT, parse_team_stats(game_id, boxscore))
    conn.execute(RAW_UPSERT, (game_id, fetched_at, payload_hash, blob))

def rebuild_box_scores_from_raw():
    """Re-parse every stored payload into Box_Scores. No network calls."""
    rows = conn.execute("SELECT game_id, payload FROM Raw_Boxscores WHERE payload IS NOT NULL").fetchall()
    for game_id, blob in rows:
        conn.execute("DELETE FROM Box_Scores WHERE game_id = ?", (game_id,))
        conn.executemany(BOXSCORE_INSERT, parse_team_stats(game_id, unpack_boxscore(blob)))
    conn.commit()
    print(f"Rebuilt Box_Scores from {len(rows)} stored payloads.")

In [73]:
def final_games_with_fetch_state():
    cur = conn.execute("""
        SELECT g.game_id, g.game_date, r.fetched_at, r.payload_hash,
               p.fetched_at AS plays_fetched_at, p.payload_hash AS plays_hash
        FROM Games g
        LEFT JOIN Raw_Boxscores r ON r.game_id = g.game_id
        LEFT JOIN Raw_Plays p ON p.game_id = g.game_id
        WHERE g.status = 'Final'""")
    cols = [c[0] for c in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]

def part_fetch_reason(fetched_at, payload_hash, game_date, now, missing_label):
    """Why one part of the summary (box score or play-by-play) needs a fetch, or None."""
    if fetched_at is None:
        return missing_label
    stale = now - datetime.fromisoformat(fetched_at) >= timedelta(hours=REFRESH_AFTER_HOURS)
    if payload_hash is None:
        return f"{missing_label}_not_published" if stale else None
    if game_date:
        kicked_off = parse_espn_date(game_date)
        if stale and now - kicked_off <= timedelta(days=RECENT_DAYS):
            return "recent_recheck"
    return None

def fetch_reason(game, now):
    """Why this Final game needs a fetch, or None to skip it. One request covers both parts."""
    return (part_fetch_reason(game["fetched_at"], game["payload_hash"], game["game_date"], now, "missing")
            or part_fetch_reason(game["plays_fetched_at"], game["plays_hash"], game["game_date"], now, "missing_plays"))

def ingest_box_scores():
    stats = Counter()
    failed = []
    now = utcnow()
 
    todo = []
    for game in final_games_with_fetch_state():
        reason = fetch_reason(game, now)
        if reason:
            todo.append((game, reason))
        else:
            stats["skipped"] += 1
 
    for game, reason in todo:
        game_id = game["game_id"]
        try:
            summary = api_get(SUMMARY_URL, params={"event": game_id})
        except requests.RequestException as exc:
            failed.append((game_id, str(exc)))
            continue
 
        boxscore = summary.get("boxscore") or {}
        stamp = iso(utcnow())
        try:
            if not boxscore.get("teams"):
                # Nothing published yet. Record the check (keeping any existing payload) so we don't re-ask right away.
                conn.execute(
                    """INSERT INTO Raw_Boxscores (game_id, fetched_at) VALUES (?, ?)
                       ON CONFLICT(game_id) DO UPDATE SET fetched_at = excluded.fetched_at""",
                    (game_id, stamp))
                stats["unchanged" if game["payload_hash"] else "no_boxscore"] += 1
            else:
                payload_hash, blob = pack_boxscore(boxscore)
                if game["payload_hash"] == payload_hash:
                    conn.execute("UPDATE Raw_Boxscores SET fetched_at = ? WHERE game_id = ?", (stamp, game_id))
                    stats["unchanged"] += 1
                else:
                    store_boxscore(game_id, boxscore, payload_hash, blob, stamp)
                    stats["updated" if game["payload_hash"] else "new"] += 1
            # Same response, so play-by-play costs no extra request (section 7).
            stats["plays_" + ingest_plays_from_summary(game_id, summary, game["plays_hash"], stamp)] += 1
            conn.commit()
        except sqlite3.Error as exc:
            conn.rollback()
            failed.append((game_id, f"database error: {exc}"))
 
    print("Box scores:", ", ".join(f"{k}={stats[k]}" for k in
          ["new", "updated", "unchanged", "no_boxscore", "skipped"]) + f", failed={len(failed)}")
    print("Play-by-play:", ", ".join(f"{k}={stats['plays_' + k]}" for k in
          ["new", "updated", "unchanged", "no_plays"]))
    if failed:
        print("Failed (retried automatically next run):", failed)
    stats["failed"] = len(failed)
    return stats

## 6. Player Boxscores

In [74]:
PLAYER_BS_INSERT = """
INSERT INTO Player_Box_Scores
(game_id, team_id, player_id, player_name, stat_category, stat_name,
 stat_label, stat_value, display_value)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

def numeric_or_none(value):
    """Keep compound ESPN display values as text rather than inventing numbers."""
    if value is None or isinstance(value, bool):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def parse_player_stats(game_id, boxscore):
    """Yield one row per game/team/player/category/statistic."""
    for team_entry in boxscore.get("players", []):
        team_id = (team_entry.get("team") or {}).get("id")
        if team_id is None:
            continue
        for category in team_entry.get("statistics", []):
            category_name = category.get("name")
            if not category_name:
                continue
            keys = category.get("keys") or []
            labels = category.get("labels") or []
            for entry in category.get("athletes", []):
                athlete = entry.get("athlete") or {}
                player_id = athlete.get("id")
                if player_id is None:
                    continue
                for index, raw_value in enumerate(entry.get("stats") or []):
                    if index >= len(keys) or not keys[index]:
                        continue
                    display = None if raw_value is None else str(raw_value)
                    yield (
                        str(game_id), str(team_id), str(player_id),
                        athlete.get("displayName"), str(category_name), str(keys[index]),
                        labels[index] if index < len(labels) else None,
                        numeric_or_none(raw_value), display,
                    )

def rebuild_player_box_scores_from_raw():
    """Completely separated: rebuilds player stats purely from the cached payloads."""
    cached = conn.execute(
        "SELECT game_id, payload FROM Raw_Boxscores WHERE payload IS NOT NULL"
    ).fetchall()
    
    total_rows = 0
    games_with_players = 0
    
    conn.execute("DELETE FROM Player_Box_Scores")
    for game_id, blob in cached:
        boxscore = unpack_boxscore(blob)
        player_rows = list(parse_player_stats(game_id, boxscore))
        if player_rows:
            games_with_players += 1
            conn.executemany(PLAYER_BS_INSERT, player_rows)
            total_rows += len(player_rows)
            
    conn.commit()
    print(f"Rebuilt Player_Box_Scores: {games_with_players} games, {total_rows} rows from {len(cached)} cached payloads.")
    return {"games": games_with_players, "rows": total_rows}

In [75]:
def store_player_boxscore(game_id, boxscore, payload_hash):
    """Write parsed player rows and update the hash tracker (caller commits)."""
    conn.execute("DELETE FROM Player_Box_Scores WHERE game_id = ?", (game_id,))
    rows = list(parse_player_stats(game_id, boxscore))
    if rows:
        conn.executemany(PLAYER_BS_INSERT, rows)
    conn.execute(
        """INSERT INTO Player_Boxscore_Fetches (game_id, payload_hash) VALUES (?, ?)
           ON CONFLICT(game_id) DO UPDATE SET payload_hash = excluded.payload_hash""",
        (game_id, payload_hash),
    )

def ingest_player_box_scores():
    """Incrementally parse only new or updated raw boxscores into Player_Box_Scores."""
    stats = Counter()
    rows = conn.execute("""
        SELECT r.game_id, r.payload_hash, r.payload, p.payload_hash AS parsed_hash
        FROM Raw_Boxscores r
        LEFT JOIN Player_Boxscore_Fetches p ON p.game_id = r.game_id
        WHERE r.payload IS NOT NULL
    """).fetchall()

    for game_id, raw_hash, blob, parsed_hash in rows:
        if raw_hash == parsed_hash:
            stats["skipped"] += 1
            continue
        store_player_boxscore(game_id, unpack_boxscore(blob), raw_hash)
        stats["updated" if parsed_hash else "new"] += 1

    conn.commit()
    print("Player box scores:", ", ".join(f"{k}={stats[k]}" for k in ["new", "updated", "skipped"]))
    return stats

## 7. Play-by-Play
The `drives` and `scoringPlays` sections of each summary are stored raw (compressed) in `Raw_Plays`, then parsed into two tables:

- `Plays`: one row per play, using `collect_drives` and `flatten_plays` from `map_schema.py`, so the notebook and the schema-mapping script share the same column definitions.
- `Scoring_Plays`: one row per score from ESPN's `scoringPlays` list. Points come from the change in the scoring team's score (so a touchdown includes its PAT or two-point try). Each score is matched by play id to the play itself, and `possession_team_id` is that play's `start.team.id`, the team that had the ball when the play began.

`is_offense_score` is 1 when the scoring team had the ball, so pick-sixes, fumble returns, punt/kickoff returns, blocked kicks and safeties are 0. Possession is used instead of ESPN's play type because the type text isn't reliable (for example, a fumble-return touchdown can be labelled `Fumble Recovery (Opponent)` and a passing touchdown `Pass Reception`). On kickoffs and punts, `start.team.id` is the kicking team, so return touchdowns are correctly not charged to the defense.

Per-play `homeScore`/`awayScore` values are sometimes recorded out of order, so points are taken from `scoringPlays` instead.

A parsing change only needs `rebuild_plays_from_raw()`, with no new requests.

In [76]:
# Columns produced by map_schema.flatten_plays, in Plays table order.
PLAY_COLUMNS = [
    "game_id", "drive_number", "drive_id", "drive_team_id", "drive_team_name", "drive_result",
    "drive_description", "play_number_in_drive", "play_id", "sequence_number", "period", "clock",
    "play_type_id", "play_type_text", "play_type_abbreviation", "text",
    "start_down", "start_distance", "start_yard_line", "start_yards_to_endzone", "start_team_id",
    "end_down", "end_distance", "end_yard_line", "end_yards_to_endzone", "end_team_id",
    "stat_yardage", "home_score", "away_score", "scoring_play", "is_penalty", "is_turnover",
    "wallclock", "modified",
]
PLAY_INSERT = (f"INSERT INTO Plays ({', '.join(PLAY_COLUMNS)}) "
               f"VALUES ({', '.join(':' + c for c in PLAY_COLUMNS)})")

SCORING_PLAY_INSERT = """
INSERT INTO Scoring_Plays (game_id, score_number, play_id, period, clock, team_id, points,
                           play_type_text, scoring_type, possession_team_id, is_offense_score, text)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

RAW_PLAYS_UPSERT = """
INSERT INTO Raw_Plays (game_id, fetched_at, payload_hash, payload) VALUES (?, ?, ?, ?)
ON CONFLICT(game_id) DO UPDATE SET
    fetched_at = excluded.fetched_at, payload_hash = excluded.payload_hash, payload = excluded.payload
"""

def plays_section(summary):
    """The part of a summary response stored in Raw_Plays, or None if ESPN has no plays yet."""
    section = {"drives": summary.get("drives") or {}, "scoringPlays": summary.get("scoringPlays") or []}
    has_plays = any(drive.get("plays") for drive in collect_drives(section))
    return section if has_plays else None

def parse_plays(game_id, section):
    """Flatten the drives into Plays rows (dicts keyed by PLAY_COLUMNS)."""
    rows = flatten_plays(str(game_id), collect_drives(section))
    return [{c: row.get(c) for c in PLAY_COLUMNS} for row in rows]

def parse_scoring_plays(game_id, section, plays, home_team_id):
    """Turn scoringPlays into Scoring_Plays rows, with points and who had the ball."""
    by_play_id = {p["play_id"]: p for p in plays if p["play_id"] is not None}
    last = {"home": 0, "away": 0}
    rows = []
    for number, score in enumerate(section.get("scoringPlays", []), start=1):
        team_id = (score.get("team") or {}).get("id")
        team_id = str(team_id) if team_id is not None else None
        side = "home" if team_id == home_team_id else "away"
        home_score, away_score = int_or_none(score.get("homeScore")), int_or_none(score.get("awayScore"))
        new_total = home_score if side == "home" else away_score
        points = None if new_total is None else new_total - last[side]
        if home_score is not None and away_score is not None:
            last = {"home": home_score, "away": away_score}

        play = by_play_id.get(str(score.get("id")))
        possession = None
        if play is not None:
            possession = play["start_team_id"] or play["drive_team_id"]
            possession = str(possession) if possession is not None else None
        is_offense = None if possession is None or team_id is None else int(possession == team_id)

        rows.append((
            str(game_id), number, str(score.get("id")), (score.get("period") or {}).get("number"),
            (score.get("clock") or {}).get("displayValue"), team_id, points,
            (score.get("type") or {}).get("text"), (score.get("scoringType") or {}).get("name"),
            possession, is_offense, score.get("text"),
        ))
    return rows

def store_plays(game_id, section, payload_hash, blob, fetched_at):
    """Write the raw play-by-play and its parsed rows together (the caller commits)."""
    home_team_id = conn.execute("SELECT home_team_id FROM Games WHERE game_id = ?", (game_id,)).fetchone()[0]
    plays = parse_plays(game_id, section)
    conn.execute("DELETE FROM Plays WHERE game_id = ?", (game_id,))
    conn.execute("DELETE FROM Scoring_Plays WHERE game_id = ?", (game_id,))
    conn.executemany(PLAY_INSERT, plays)
    conn.executemany(SCORING_PLAY_INSERT, parse_scoring_plays(game_id, section, plays, home_team_id))
    if blob is not None:
        conn.execute(RAW_PLAYS_UPSERT, (game_id, fetched_at, payload_hash, blob))

def ingest_plays_from_summary(game_id, summary, stored_hash, fetched_at):
    """Store play-by-play from a summary response already fetched for the box score.
    Returns new / updated / unchanged / no_plays. The caller commits."""
    section = plays_section(summary)
    if section is None:
        # Nothing published yet. Record the check (keeping any existing payload).
        conn.execute(
            """INSERT INTO Raw_Plays (game_id, fetched_at) VALUES (?, ?)
               ON CONFLICT(game_id) DO UPDATE SET fetched_at = excluded.fetched_at""",
            (game_id, fetched_at))
        return "unchanged" if stored_hash else "no_plays"
    payload_hash, blob = pack_boxscore(section)  # same canonical-JSON hashing and compression
    if payload_hash == stored_hash:
        conn.execute("UPDATE Raw_Plays SET fetched_at = ? WHERE game_id = ?", (fetched_at, game_id))
        return "unchanged"
    store_plays(game_id, section, payload_hash, blob, fetched_at)
    return "updated" if stored_hash else "new"

def rebuild_plays_from_raw():
    """Re-parse every stored play-by-play payload into Plays and Scoring_Plays. No network calls."""
    cached = conn.execute("SELECT game_id, payload FROM Raw_Plays WHERE payload IS NOT NULL").fetchall()
    conn.execute("DELETE FROM Plays")
    conn.execute("DELETE FROM Scoring_Plays")
    for game_id, blob in cached:
        store_plays(game_id, unpack_boxscore(blob), None, None, None)
    conn.commit()
    print(f"Rebuilt Plays and Scoring_Plays from {len(cached)} stored payloads.")

## 8. Team Defensive Stats
Sums each player stat in the `defensive` and `interceptions` categories to one total per game and team. This only reads `Player_Box_Scores` (no ESPN requests), so the table is fully rebuilt every run and can never hold stale or duplicate rows.

`team_total` stays `NULL` when every player value for that stat is `NULL` (e.g. compound display values), rather than becoming `0`.

In [77]:
DEFENSIVE_CATEGORIES = ("defensive", "interceptions")

def build_team_defensive_stats():
    """Rebuild Team_Defensive_Stats from the player-level defensive stats."""
    player_stats = pd.read_sql_query(
        f"""
        SELECT game_id, team_id, stat_category, stat_name, stat_value
        FROM Player_Box_Scores
        WHERE stat_category IN ({",".join("?" * len(DEFENSIVE_CATEGORIES))})
        """,
        conn,
        params=DEFENSIVE_CATEGORIES,
    )

    # Sum numeric statistics for each team in each game.
    # min_count=1 preserves NULL when all values are missing.
    team_stats = (
        player_stats.groupby(["game_id", "team_id", "stat_category", "stat_name"], as_index=False)["stat_value"]
        .sum(min_count=1)
        .rename(columns={"stat_value": "team_total"})
    )

    # Replace previously calculated results so reruns don't create duplicate records.
    conn.execute("DELETE FROM Team_Defensive_Stats")
    team_stats.to_sql("Team_Defensive_Stats", conn, if_exists="append", index=False)
    conn.commit()

    print(f"Team defensive stats: player_rows={len(player_stats):,}, team_rows={len(team_stats):,}, "
          f"games={team_stats['game_id'].nunique()}, teams={team_stats['team_id'].nunique()}")
    return {"player_rows": len(player_stats), "team_rows": len(team_stats)}

## 9. Team Defense from Opponent Box Scores
A second, independent view of team defense, kept separate from the player-stat sums in section 8. In every game, the opponent's offensive team stats in `Box_Scores` are exactly what this team's defense allowed, so each opponent row is copied to the defending team, with `opponent_id` recording who that offense was.

The stats are renamed from the defense's point of view (`DEFENSE_STAT_NAMES`), and values are unchanged:

| Opponent stat (`Box_Scores`) | Defensive stat | Meaning |
| --- | --- | --- |
| `totalYards`, `netPassingYards`, `rushingYards` | `totalYardsAllowed`, `passingYardsAllowed`, `rushingYardsAllowed` | Yards given up |
| `yardsPerPass`, `yardsPerRushAttempt` | `yardsPerPassAllowed`, `yardsPerRushAllowed` | Yards per play given up |
| `firstDowns`, `thirdDownEff`, `fourthDownEff` | `firstDownsAllowed`, `thirdDownEffAllowed`, `fourthDownEffAllowed` | Drives the defense let continue |
| `completionAttempts`, `rushingAttempts` | `completionsAttemptsAllowed`, `rushingAttemptsFaced` | Opponent volume |
| `turnovers`, `interceptions`, `fumblesLost` | `takeaways`, `interceptionsMade`, `fumblesRecovered` | Takeaways forced by this defense |
| `possessionTime`, `totalPenaltiesYards` | `opponentPossessionTime`, `opponentPenaltiesYards` | Opponent context, not strictly defense |
| *(from `Games`)* | `pointsAllowed` | Opponent's final score |
| *(from `Scoring_Plays`)* | `offensePointsAllowed` | Points the opponent scored while it had the ball (no defensive or return touchdowns, no safeties) |

A stat ESPN adds later that is not in the mapping is stored as `opponent<StatName>` (e.g. `opponentSacks`) so it can never be mistaken for an offensive stat. Compound values such as `completionsAttemptsAllowed` keep `stat_value = NULL` with the text in `display_value`, as in `Box_Scores`.

`pointsAllowed` is not in ESPN's team box score, so it is added from the opponent's score in `Games` for Final games.

`offensePointsAllowed` is only written for games whose play-by-play is stored, every score could be matched to a play, and the scores add up to the final score in `Games`. Anything else is left out rather than guessed, so rankings fall back to `pointsAllowed` for that game.

In [78]:
# Opponent's Box_Scores stat -> (defensive stat name, label), from the defending team's point of view.
DEFENSE_STAT_NAMES = {
    "totalYards":          ("totalYardsAllowed",          "Total yards allowed"),
    "netPassingYards":     ("passingYardsAllowed",        "Passing yards allowed"),
    "rushingYards":        ("rushingYardsAllowed",        "Rushing yards allowed"),
    "yardsPerPass":        ("yardsPerPassAllowed",        "Yards per pass allowed"),
    "yardsPerRushAttempt": ("yardsPerRushAllowed",        "Yards per rush allowed"),
    "firstDowns":          ("firstDownsAllowed",          "1st downs allowed"),
    "thirdDownEff":        ("thirdDownEffAllowed",        "3rd down efficiency allowed"),
    "fourthDownEff":       ("fourthDownEffAllowed",       "4th down efficiency allowed"),
    "completionAttempts":  ("completionsAttemptsAllowed", "Comp/Att allowed"),
    "rushingAttempts":     ("rushingAttemptsFaced",       "Rushing attempts faced"),
    "turnovers":           ("takeaways",                  "Takeaways"),
    "interceptions":       ("interceptionsMade",          "Interceptions made"),
    "fumblesLost":         ("fumblesRecovered",           "Fumbles recovered"),
    "possessionTime":      ("opponentPossessionTime",     "Opponent possession"),
    "totalPenaltiesYards": ("opponentPenaltiesYards",     "Opponent penalties"),
}
POINTS_ALLOWED = ("pointsAllowed", "Points allowed")
OFFENSE_POINTS_ALLOWED = ("offensePointsAllowed", "Points allowed to offense")

def defense_names_cte():
    """SQL CTE `defense_names(stat_name, defense_name, defense_label)` plus its parameters."""
    rows = ", ".join("(?, ?, ?)" for _ in DEFENSE_STAT_NAMES)
    params = [value for stat, (name, label) in DEFENSE_STAT_NAMES.items() for value in (stat, name, label)]
    return f"WITH defense_names(stat_name, defense_name, defense_label) AS (VALUES {rows})", params

# Unmapped stats fall back to opponent<StatName> / "Opponent <label>".
DEFENSE_NAME_SQL = "COALESCE(n.defense_name, 'opponent' || UPPER(SUBSTR(b.stat_name, 1, 1)) || SUBSTR(b.stat_name, 2))"
DEFENSE_LABEL_SQL = "COALESCE(n.defense_label, 'Opponent ' || b.stat_label)"

OPPONENT_DEFENSE_INSERT = f"""
INSERT INTO Team_Defense_Opponent_Stats
    (game_id, team_id, opponent_id, stat_name, stat_label, stat_value, display_value)
{{cte}}
SELECT b.game_id,
       CASE WHEN b.team_id = g.home_team_id THEN g.away_team_id ELSE g.home_team_id END,
       b.team_id, {DEFENSE_NAME_SQL}, {DEFENSE_LABEL_SQL}, b.stat_value, b.display_value
FROM Box_Scores b
JOIN Games g ON g.game_id = b.game_id
LEFT JOIN defense_names n ON n.stat_name = b.stat_name
WHERE b.team_id IN (g.home_team_id, g.away_team_id)
"""

POINTS_ALLOWED_INSERT = """
INSERT INTO Team_Defense_Opponent_Stats
    (game_id, team_id, opponent_id, stat_name, stat_label, stat_value, display_value)
SELECT game_id, home_team_id, away_team_id, ?, ?, away_score, CAST(away_score AS TEXT)
FROM Games WHERE status = 'Final' AND away_score IS NOT NULL
UNION ALL
SELECT game_id, away_team_id, home_team_id, ?, ?, home_score, CAST(home_score AS TEXT)
FROM Games WHERE status = 'Final' AND home_score IS NOT NULL
"""

# Points the opposing offense scored with the ball, for games whose scoring plays are complete
# (all matched to a play, and adding up to the Games final score).
OFFENSE_POINTS_INSERT = """
INSERT INTO Team_Defense_Opponent_Stats
    (game_id, team_id, opponent_id, stat_name, stat_label, stat_value, display_value)
WITH sides AS (
    SELECT game_id, away_team_id AS defense, home_team_id AS offense, home_score AS offense_score
    FROM Games WHERE status = 'Final' AND home_score IS NOT NULL
    UNION ALL
    SELECT game_id, home_team_id, away_team_id, away_score
    FROM Games WHERE status = 'Final' AND away_score IS NOT NULL
), scored AS (
    SELECT game_id, team_id,
           SUM(points) AS total_points,
           SUM(CASE WHEN is_offense_score = 1 THEN points ELSE 0 END) AS offense_points,
           SUM(is_offense_score IS NULL OR points IS NULL) AS unmatched
    FROM Scoring_Plays GROUP BY game_id, team_id
)
SELECT s.game_id, s.defense, s.offense, ?, ?,
       COALESCE(sc.offense_points, 0), CAST(COALESCE(sc.offense_points, 0) AS TEXT)
FROM sides s
JOIN Raw_Plays r ON r.game_id = s.game_id AND r.payload IS NOT NULL
LEFT JOIN scored sc ON sc.game_id = s.game_id AND sc.team_id = s.offense
WHERE COALESCE(sc.total_points, 0) = s.offense_score AND COALESCE(sc.unmatched, 0) = 0
"""

def build_team_defense_opponent_stats():
    """Rebuild Team_Defense_Opponent_Stats from the opponents' team box scores and final scores."""
    cte, params = defense_names_cte()
    with conn:
        # Full rebuild: corrected or removed box scores cannot leave stale rows behind.
        conn.execute("DELETE FROM Team_Defense_Opponent_Stats")
        box_rows = conn.execute(OPPONENT_DEFENSE_INSERT.format(cte=cte), params).rowcount
        point_rows = conn.execute(POINTS_ALLOWED_INSERT, POINTS_ALLOWED * 2).rowcount
        offense_point_rows = conn.execute(OFFENSE_POINTS_INSERT, OFFENSE_POINTS_ALLOWED).rowcount
    games, teams = conn.execute(
        "SELECT COUNT(DISTINCT game_id), COUNT(DISTINCT team_id) FROM Team_Defense_Opponent_Stats"
    ).fetchone()
    unmapped = conn.execute(
        f"SELECT DISTINCT stat_name FROM Box_Scores WHERE stat_name NOT IN ({','.join('?' * len(DEFENSE_STAT_NAMES))})",
        list(DEFENSE_STAT_NAMES),
    ).fetchall()
    print(f"Team defense from opponents: box_score_rows={box_rows:,}, points_rows={point_rows:,}, "
          f"offense_points_rows={offense_point_rows:,}, games={games}, teams={teams}")
    if unmapped:
        print("  Not in DEFENSE_STAT_NAMES (stored as opponent<StatName>):", [s for (s,) in unmapped])
    return {"box_score_rows": box_rows, "points_rows": point_rows, "offense_points_rows": offense_point_rows}

## 10. Update the database

Safe to run as often as you like. Prints how many ESPN requests the run made; a run with nothing new should make none.

In [79]:
def update_database(force_teams=False):
    start_calls = api_calls
    load_fbs_teams(force=force_teams)
    schedule_stats = refresh_schedule()
    box_stats = ingest_box_scores()
    player_box_stats = ingest_player_box_scores()
    defensive_stats = build_team_defensive_stats()
    opponent_defense_stats = build_team_defense_opponent_stats()
    
    calls = api_calls - start_calls
    print(f"ESPN requests this run: {calls}")
    return {
        "schedule": schedule_stats, 
        "box_scores": box_stats, 
        "player_box_scores": player_box_stats, 
        "team_defensive_stats": defensive_stats,
        "team_defense_opponent_stats": opponent_defense_stats,
        "api_calls": calls
    }
 
update_database()

Teams: 136 FBS teams already stored, no fetch needed.
Schedule: days_fetched=0, new_games=0, changed_games=0, failed_days=0
Box scores: new=0, updated=0, unchanged=0, no_boxscore=0, skipped=958, failed=0
Play-by-play: new=0, updated=0, unchanged=0, no_plays=0
Player box scores: new=0, updated=0, skipped=958
Team defensive stats: player_rows=286,136, team_rows=16,087, games=955, teams=234
Team defense from opponents: box_score_rows=28,740, points_rows=1,916, offense_points_rows=1,893, games=958, teams=236
ESPN requests this run: 0


{'schedule': Counter(),
 'box_scores': Counter({'skipped': 958, 'failed': 0}),
 'player_box_scores': Counter({'skipped': 958}),
 'team_defensive_stats': {'player_rows': 286136, 'team_rows': 16087},
 'team_defense_opponent_stats': {'box_score_rows': 28740,
  'points_rows': 1916,
  'offense_points_rows': 1893},
 'api_calls': 0}

## 11. Checks

In [80]:
final_not_stored = conn.execute("""
    SELECT COUNT(*) FROM Games g
    LEFT JOIN Raw_Boxscores r ON r.game_id = g.game_id
    WHERE g.status = 'Final' AND r.payload_hash IS NULL""").fetchone()[0]
print("Final games without a stored box score:", final_not_stored)
 
print("FBS teams with no games:", conn.execute("""
    SELECT t.display_name FROM Teams t
    WHERE t.conference IS NOT NULL AND t.conference != ''
      AND NOT EXISTS (SELECT 1 FROM Games g WHERE t.team_id IN (g.home_team_id, g.away_team_id))
""").fetchall())
 
print("Foreign key violations:", conn.execute("PRAGMA foreign_key_check").fetchall())
 
print("Row counts (Teams, Games, Box_Scores, Raw_Boxscores, Player_Box_Scores, Team_Defensive_Stats, Team_Defense_Opponent_Stats, Raw_Plays, Plays, Scoring_Plays):", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Teams), (SELECT COUNT(*) FROM Games),
           (SELECT COUNT(*) FROM Box_Scores), (SELECT COUNT(*) FROM Raw_Boxscores),
           (SELECT COUNT(*) FROM Player_Box_Scores), (SELECT COUNT(*) FROM Team_Defensive_Stats),
           (SELECT COUNT(*) FROM Team_Defense_Opponent_Stats), (SELECT COUNT(*) FROM Raw_Plays),
           (SELECT COUNT(*) FROM Plays), (SELECT COUNT(*) FROM Scoring_Plays)""").fetchone())

Final games without a stored box score: 0
FBS teams with no games: []
Foreign key violations: []
Row counts (Teams, Games, Box_Scores, Raw_Boxscores, Player_Box_Scores, Team_Defensive_Stats, Team_Defense_Opponent_Stats, Raw_Plays, Plays, Scoring_Plays): (236, 958, 28740, 958, 479413, 16087, 32549, 958, 170611, 8596)


A stat should only have fewer numeric rows than total rows if it is a compound value (for example `completionAttempts`, `totalPenaltiesYards`) or, for `fourthDownEff`, a team with no attempts.

In [81]:
pd.read_sql_query("""
SELECT stat_name, COUNT(*) AS rows_, COUNT(stat_value) AS numeric_rows
FROM Box_Scores
GROUP BY stat_name
ORDER BY stat_name
""", conn)

,stat_name,rows_,numeric_rows
0,completionAttempts,1916,0
1,firstDowns,1916,1916
2,fourthDownEff,1916,1608
3,fumblesLost,1916,1916
4,interceptions,1916,1916
5,netPassingYards,1916,1916
6,possessionTime,1916,1916
7,rushingAttempts,1916,1916
8,rushingYards,1916,1916
9,thirdDownEff,1916,1916


Spot check of team defensive totals: Tennessee Volunteers (team `2633`) in game `401856745` (Kentucky at Tennessee, Nov 7).

In [82]:
pd.read_sql_query("""
SELECT stat_category, stat_name, team_total
FROM Team_Defensive_Stats
WHERE game_id = '401856745' AND team_id = '2633'
ORDER BY stat_category, stat_name
""", conn)

,stat_category,stat_name,team_total


Each `Box_Scores` row should appear exactly once as an opponent stat (so the first two counts match and nothing is unmatched), and every Final game should give both teams a `pointsAllowed` row.

In [83]:
print("Box_Scores rows vs. opponent-defense rows (excluding points):", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Box_Scores),
           (SELECT COUNT(*) FROM Team_Defense_Opponent_Stats WHERE stat_name NOT IN (?, ?))""",
    (POINTS_ALLOWED[0], OFFENSE_POINTS_ALLOWED[0])).fetchone())

cte, params = defense_names_cte()
print("Box_Scores rows with no matching opponent-defense row:", conn.execute(f"""
    {cte}
    SELECT COUNT(*) FROM Box_Scores b
    LEFT JOIN defense_names n ON n.stat_name = b.stat_name
    WHERE NOT EXISTS (SELECT 1 FROM Team_Defense_Opponent_Stats d
                      WHERE d.game_id = b.game_id AND d.opponent_id = b.team_id
                        AND d.stat_name = {DEFENSE_NAME_SQL}
                        AND d.stat_value IS b.stat_value AND d.display_value IS b.display_value)
""", params).fetchone()[0])

print("Final games vs. points-allowed rows:", conn.execute("""
    SELECT (SELECT 2 * COUNT(*) FROM Games WHERE status = 'Final'),
           (SELECT COUNT(*) FROM Team_Defense_Opponent_Stats WHERE stat_name = ?)""",
    (POINTS_ALLOWED[0],)).fetchone())

Box_Scores rows vs. opponent-defense rows (excluding points): (28740, 28740)
Box_Scores rows with no matching opponent-defense row: 0
Final games vs. points-allowed rows: (1916, 1916)


Spot check of the opponent-based view for the same game: what Tennessee's defense allowed against Kentucky.

In [84]:
pd.read_sql_query("""
SELECT t.display_name AS team, o.display_name AS opponent, d.stat_name, d.stat_label, d.stat_value, d.display_value
FROM Team_Defense_Opponent_Stats d
JOIN Teams t ON t.team_id = d.team_id
JOIN Teams o ON o.team_id = d.opponent_id
WHERE d.game_id = '401856745' AND d.team_id = '2633'
ORDER BY d.stat_name
""", conn)

,team,opponent,stat_name,stat_label,stat_value,display_value


Play-by-play checks. Every Final game should have stored plays. For each game the scoring plays should add up to the final score in `Games`; any game that doesn't is listed, and has no `offensePointsAllowed` rows, so rankings use `pointsAllowed` for it. The last table lists every score that wasn't charged to the defense, to spot-check the possession rule.

In [85]:
from IPython.display import Markdown

print("Final games vs. games with stored play-by-play:", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Games WHERE status = 'Final'),
           (SELECT COUNT(*) FROM Raw_Plays r JOIN Games g ON g.game_id = r.game_id
            WHERE g.status = 'Final' AND r.payload IS NOT NULL)""").fetchone())

print("Games with play-by-play vs. games with offensePointsAllowed:", conn.execute("""
    SELECT (SELECT COUNT(*) FROM Raw_Plays WHERE payload IS NOT NULL),
           (SELECT COUNT(DISTINCT game_id) FROM Team_Defense_Opponent_Stats WHERE stat_name = ?)""",
    (OFFENSE_POINTS_ALLOWED[0],)).fetchone())

print("Scoring plays not matched to a play:", conn.execute(
    "SELECT COUNT(*) FROM Scoring_Plays WHERE is_offense_score IS NULL").fetchone()[0])

display(Markdown("**Games whose scoring plays don't add up to the final score**"))
display(pd.read_sql_query("""
WITH totals AS (
    SELECT game_id, team_id, SUM(points) AS scored FROM Scoring_Plays GROUP BY game_id, team_id
)
SELECT g.game_id, g.name, g.home_score, COALESCE(h.scored, 0) AS home_from_plays,
       g.away_score, COALESCE(a.scored, 0) AS away_from_plays
FROM Games g
JOIN Raw_Plays r ON r.game_id = g.game_id AND r.payload IS NOT NULL
LEFT JOIN totals h ON h.game_id = g.game_id AND h.team_id = g.home_team_id
LEFT JOIN totals a ON a.game_id = g.game_id AND a.team_id = g.away_team_id
WHERE g.status = 'Final'
  AND (COALESCE(h.scored, 0) != g.home_score OR COALESCE(a.scored, 0) != g.away_score)
""", conn))

display(Markdown("**Scores not charged to the defense, by ESPN play type**"))
display(pd.read_sql_query("""
SELECT play_type_text, COUNT(*) AS scores, SUM(points) AS points
FROM Scoring_Plays WHERE is_offense_score = 0
GROUP BY play_type_text ORDER BY scores DESC
""", conn))

Final games vs. games with stored play-by-play: (958, 956)
Games with play-by-play vs. games with offensePointsAllowed: (956, 954)
Scoring plays not matched to a play: 8


**Games whose scoring plays don't add up to the final score**

,game_id,name,home_score,home_from_plays,away_score,away_from_plays
0,401754521,East Texas A&M Lions at SMU Mustangs,42,48,13,13
1,401756879,Weber State Wildcats at Arizona Wildcats,48,0,3,0
2,401760376,UNLV Rebels at Miami (OH) RedHawks,38,37,41,55
3,401762831,Eastern Michigan Eagles at Buffalo Bulls,31,31,30,29
4,401762837,Massachusetts Minutemen at Kent State Golden F...,42,41,6,6
5,401757301,Liberty Flames at Florida International Panthers,34,54,27,27
6,401767134,Sacramento State Hornets at UC Davis Aggies,31,28,27,27
7,401756966,Iowa State Cyclones at Oklahoma State Cowboys,13,13,20,13
8,401761667,South Alabama Jaguars at Texas State Bobcats,49,43,26,26
9,401752923,Maryland Terrapins at Michigan State Spartans,38,38,28,21


**Scores not charged to the defense, by ESPN play type**

,play_type_text,scores,points
0,Interception Return Touchdown,131,896
1,Fumble Return Touchdown,54,378
2,Safety,44,86
3,Kickoff Return Touchdown,44,308
4,Punt,22,151
5,Blocked Punt Touchdown,20,132
6,Punt Return Touchdown,14,99
7,Fumble Recovery (Opponent),11,76
8,Blocked Punt,9,48
9,Pass Interception Return,6,43


Show what Tennessee's defense has done across the whole year

In [86]:
from IPython.display import Markdown, display

TEAM_ID = "2633"  # Tennessee Volunteers

# Games played: a category missing from a game means zero (e.g. no interceptions),
# so totals are divided by every Final game the team played, not just games with rows.
games_played = conn.execute(
    "SELECT COUNT(*) FROM Games WHERE status = 'Final' AND ? IN (home_team_id, away_team_id)",
    (TEAM_ID,),
).fetchone()[0]

# Opponent-based view. Efficiency stats ("9-13") are a season rate: total conversions / total attempts.
opponent_avgs = pd.read_sql_query("""
SELECT stat_name, MAX(stat_label) AS stat_label,
       CASE WHEN stat_name LIKE '%EffAllowed'
            THEN 1.0 * SUM(CAST(SUBSTR(display_value, 1, INSTR(display_value, '-') - 1) AS INTEGER))
                 / NULLIF(SUM(CAST(SUBSTR(display_value, INSTR(display_value, '-') + 1) AS INTEGER)), 0)
            ELSE SUM(stat_value) / :games END AS season_value
FROM Team_Defense_Opponent_Stats
WHERE team_id = :team AND (stat_value IS NOT NULL OR stat_name LIKE '%EffAllowed')
GROUP BY stat_name
ORDER BY stat_name
""", conn, params={"team": TEAM_ID, "games": games_played})
opponent_avgs["measure"] = opponent_avgs["stat_name"].map(
    lambda s: "season rate" if s.endswith("EffAllowed") else "per game")

# Player-stat view: team totals summed over the season, divided by games played.
player_avgs = pd.read_sql_query("""
SELECT stat_category, stat_name, SUM(team_total) / :games AS per_game
FROM Team_Defensive_Stats
WHERE team_id = :team
GROUP BY stat_category, stat_name
ORDER BY stat_category, stat_name
""", conn, params={"team": TEAM_ID, "games": games_played})

team_name = conn.execute("SELECT display_name FROM Teams WHERE team_id = ?", (TEAM_ID,)).fetchone()[0]
display(Markdown(f"#### {team_name} defense, {YEAR} season ({games_played} games)"))
display(Markdown("**Allowed / forced, from opponent box scores** (`opponentPossessionTime` is in seconds)"))
display(opponent_avgs.set_index("stat_name").round(3))
display(Markdown("**Per game, from player defensive stats**"))
display(player_avgs.set_index(["stat_category", "stat_name"]).round(2))

#### Tennessee Volunteers defense, 2025 season (13 games)

**Allowed / forced, from opponent box scores** (`opponentPossessionTime` is in seconds)

,stat_label,season_value,measure
stat_name,,,
firstDownsAllowed,1st downs allowed,22.462,per game
fourthDownEffAllowed,4th down efficiency allowed,0.520,season rate
fumblesRecovered,Fumbles recovered,0.769,per game
interceptionsMade,Interceptions made,0.769,per game
offensePointsAllowed,Points allowed to offense,27.077,per game
opponentPossessionTime,Opponent possession,1971.000,per game
passingYardsAllowed,Passing yards allowed,244.538,per game
pointsAllowed,Points allowed,28.846,per game
rushingAttemptsFaced,Rushing attempts faced,37.538,per game


**Per game, from player defensive stats**

per_game
stat_category stat_name                       
defensive     defensiveTouchdowns         0.46
              hurries                     3.00
              passesDefended              4.08
              sacks                       2.85
              soloTackles                34.00
              tacklesForLoss              6.69
              totalTackles               74.92
interceptions interceptionTouchdowns      0.23
              interceptionYards          14.23
              interceptions               0.77

Raw defensive ranking of every FBS team (no opponent adjustment). Scoring defense uses `offensePointsAllowed` (points scored by the opposing offense), falling back to `pointsAllowed` for any game without usable play-by-play. Each stat is a per-game average or season rate, converted to a z-score across FBS teams so that higher is better, then averaged using the weights in `RANKING_STATS`. The full table is stored in `defense_rankings`.

In [87]:
# Raw (not opponent-adjusted) defensive ranking of FBS teams.
# Each stat becomes a per-game average (or a season rate for 3rd-down %), is turned
# into a z-score across FBS teams, and flipped so that higher always means better defense.
# The composite score is the weighted mean of those z-scores.

# stat -> (weight, higher_is_better). Takeaways already includes INTs + fumbles recovered,
# so those two are left out to avoid counting them twice.
RANKING_STATS = {
    "offensePointsAllowed":(1.0, False),
    "totalYardsAllowed":   (1.0, False),
    "yardsPerPassAllowed": (1.0, False),
    "yardsPerRushAllowed": (1.0, False),
    "firstDownsAllowed":   (1.0, False),
    "thirdDownRateAllowed":(1.0, False),
    "takeaways":           (1.0, True),
    "sacks":               (1.0, True),
    "tacklesForLoss":      (1.0, True),
    "passesDefended":      (1.0, True),
}

# Final games played per FBS team (the per-game denominator).
fbs_games = pd.read_sql_query("""
SELECT t.team_id, t.display_name AS team, t.conference, COUNT(g.game_id) AS games
FROM Teams t
JOIN Games g ON g.status = 'Final' AND t.team_id IN (g.home_team_id, g.away_team_id)
WHERE t.conference IS NOT NULL AND t.conference != ''
GROUP BY t.team_id
""", conn).set_index("team_id")

# Opponent-based totals; 3rd down comes from "made-attempts" display values.
opp = pd.read_sql_query("""
SELECT team_id,
       SUM(CASE WHEN stat_name = 'pointsAllowed'       THEN stat_value END) AS pointsAllowed,
       SUM(CASE WHEN stat_name = 'totalYardsAllowed'   THEN stat_value END) AS totalYardsAllowed,
       SUM(CASE WHEN stat_name = 'passingYardsAllowed' THEN stat_value END) AS passingYardsAllowed,
       SUM(CASE WHEN stat_name = 'rushingYardsAllowed' THEN stat_value END) AS rushingYardsAllowed,
       SUM(CASE WHEN stat_name = 'rushingAttemptsFaced' THEN stat_value END) AS rushingAttemptsFaced,
       SUM(CASE WHEN stat_name = 'firstDownsAllowed'   THEN stat_value END) AS firstDownsAllowed,
       SUM(CASE WHEN stat_name = 'takeaways'           THEN stat_value END) AS takeaways,
       SUM(CASE WHEN stat_name = 'completionsAttemptsAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '/') + 1) AS INTEGER) END) AS passAttemptsFaced,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, 1, INSTR(display_value, '-') - 1) AS INTEGER) END) AS thirdDownConv,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '-') + 1) AS INTEGER) END) AS thirdDownAtt
FROM Team_Defense_Opponent_Stats
GROUP BY team_id
""", conn).set_index("team_id")

# Points scored by the opposing offense; games without usable play-by-play fall back to all points.
points = pd.read_sql_query("""
SELECT team_id,
       SUM(COALESCE(offense_points, all_points)) AS offensePointsAllowed,
       SUM(offense_points IS NULL) AS games_without_pbp
FROM (
    SELECT game_id, team_id,
           MAX(CASE WHEN stat_name = 'pointsAllowed'        THEN stat_value END) AS all_points,
           MAX(CASE WHEN stat_name = 'offensePointsAllowed' THEN stat_value END) AS offense_points
    FROM Team_Defense_Opponent_Stats
    WHERE stat_name IN ('pointsAllowed', 'offensePointsAllowed')
    GROUP BY game_id, team_id
)
GROUP BY team_id
""", conn).set_index("team_id")

# Player-stat totals summed to the team.
player = pd.read_sql_query("""
SELECT team_id, stat_name, SUM(team_total) AS total
FROM Team_Defensive_Stats
WHERE stat_name IN ('sacks', 'tacklesForLoss', 'passesDefended')
GROUP BY team_id, stat_name
""", conn).pivot(index="team_id", columns="stat_name", values="total")

season = fbs_games.join(opp).join(points).join(player).fillna(0)
g = season["games"]
season_stats = pd.DataFrame({
    "team": season["team"],
    "conference": season["conference"],
    "games": g,
    "pointsAllowed":        season["pointsAllowed"] / g,
    "offensePointsAllowed": season["offensePointsAllowed"] / g,
    "totalYardsAllowed":    season["totalYardsAllowed"] / g,
    # Season rates (total yards / total attempts) rather than averaging per-game rates.
    "yardsPerPassAllowed":  season["passingYardsAllowed"] / season["passAttemptsFaced"],
    "yardsPerRushAllowed":  season["rushingYardsAllowed"] / season["rushingAttemptsFaced"],
    "firstDownsAllowed":    season["firstDownsAllowed"] / g,
    "thirdDownRateAllowed": season["thirdDownConv"] / season["thirdDownAtt"],
    "takeaways":            season["takeaways"] / g,
    "sacks":                season["sacks"] / g,
    "tacklesForLoss":       season["tacklesForLoss"] / g,
    "passesDefended":       season["passesDefended"] / g,
})

# z-score each stat, flip "lower is better" stats, then take the weighted mean.
weights = pd.Series({s: w for s, (w, _) in RANKING_STATS.items()})
signs = pd.Series({s: 1 if better else -1 for s, (_, better) in RANKING_STATS.items()})
stat_cols = list(RANKING_STATS)
z = (season_stats[stat_cols] - season_stats[stat_cols].mean()) / season_stats[stat_cols].std()
season_stats["defense_score"] = (z * signs * weights).sum(axis=1) / weights.sum()

defense_rankings = (
    season_stats.sort_values("defense_score", ascending=False)
    .reset_index()
    .rename_axis("rank")
)
defense_rankings.index += 1

missing_pbp = int(season["games_without_pbp"].sum())
if missing_pbp:
    print(f"Note: {missing_pbp} team-games have no usable play-by-play; they use pointsAllowed instead.")

display(Markdown(f"#### Raw defensive rankings, {YEAR} season ({len(defense_rankings)} FBS teams) — top 10"))
display(defense_rankings.head(10)[[
    "team", "conference", "games", "defense_score", "offensePointsAllowed", "pointsAllowed", "totalYardsAllowed",
    "yardsPerPassAllowed", "yardsPerRushAllowed", "thirdDownRateAllowed",
    "takeaways", "sacks", "tacklesForLoss",
]].round(2))

Note: 18 team-games have no usable play-by-play; they use pointsAllowed instead.


#### Raw defensive rankings, 2025 season (136 FBS teams) — top 10

,team,conference,games,defense_score,offensePointsAllowed,pointsAllowed,totalYardsAllowed,yardsPerPassAllowed,yardsPerRushAllowed,thirdDownRateAllowed,takeaways,sacks,tacklesForLoss
rank,,,,,,,,,,,,,
1,Texas Tech Red Raiders,Big 12 Conference,14,2.06,11.64,11.79,258.29,5.51,2.20,0.30,2.29,2.93,7.57
2,Oklahoma Sooners,Southeastern Conference,13,1.68,14.23,15.46,272.54,6.24,2.39,0.28,1.00,3.46,9.46
3,Indiana Hoosiers,Big Ten Conference,16,1.62,11.12,11.69,266.00,6.20,2.99,0.30,1.88,2.88,8.06
4,Toledo Rockets,Mid-American Conference,13,1.62,13.31,13.31,254.31,5.82,2.68,0.29,1.62,2.85,7.38
5,James Madison Dukes,Sun Belt Conference,14,1.34,16.93,18.36,266.64,6.04,2.80,0.29,1.21,2.57,6.21
6,Ohio State Buckeyes,Big Ten Conference,14,1.25,8.79,9.29,219.07,5.31,2.94,0.31,1.07,2.50,5.57
7,Miami Hurricanes,Atlantic Coast Conference,16,1.24,14.38,14.81,294.12,6.29,2.95,0.31,1.56,3.12,6.38
8,Texas A&M Aggies,Southeastern Conference,13,1.17,19.77,21.00,307.38,6.03,4.19,0.22,0.77,3.31,8.46
9,Notre Dame Fighting Irish,FBS Independents,12,1.10,17.08,17.58,313.50,6.20,2.99,0.35,1.92,2.83,5.83


Opponent-adjusted defensive ranking. Each stat is fit with a ridge regression across every game (`value = league average + offense strength - defense strength`), so each defense is judged against what its opponents' offenses usually do. Uses `RANKING_STATS` and `defense_rankings` from the cell above; the full table is stored in `adjusted_defense_rankings`.

In [88]:
import numpy as np

# Opponent-adjusted defensive ranking.
# For every stat, each game is modeled as
#     value = league_avg + offense_strength[offense] - defense_strength[defense]
# and solved for all teams at once with ridge-regularized least squares, so a
# defense gets credit for holding a good offense down and less credit against a weak one.
# FCS teams only appear against FBS opponents, so they are pooled into a single "FCS" team.
# Rate stats (yards per pass/rush, 3rd-down %) are weighted by attempts in that game.
# The defense_strength values are in the stat's own units: e.g. offensePointsAllowed = 7.0 means
# "allows 7 fewer points per game than an average FBS defense would against the same schedule".
# offensePointsAllowed falls back to pointsAllowed for any game without usable play-by-play.
# The composite score reuses RANKING_STATS from the raw ranking above.

RIDGE_PENALTY = 2.0  # shrinks every team toward average by roughly 2 games' worth of evidence

fbs_ids = set(fbs_games.index)

# One row per (game, defense): what that defense allowed or produced.
game_opp = pd.read_sql_query("""
SELECT game_id, team_id AS defense, opponent_id AS offense,
       SUM(CASE WHEN stat_name = 'pointsAllowed'       THEN stat_value END) AS pointsAllowed,
       SUM(CASE WHEN stat_name = 'offensePointsAllowed' THEN stat_value END) AS offensePointsAllowed,
       SUM(CASE WHEN stat_name = 'totalYardsAllowed'   THEN stat_value END) AS totalYardsAllowed,
       SUM(CASE WHEN stat_name = 'passingYardsAllowed' THEN stat_value END) AS passingYards,
       SUM(CASE WHEN stat_name = 'rushingYardsAllowed' THEN stat_value END) AS rushingYards,
       SUM(CASE WHEN stat_name = 'rushingAttemptsFaced' THEN stat_value END) AS rushAttempts,
       SUM(CASE WHEN stat_name = 'firstDownsAllowed'   THEN stat_value END) AS firstDownsAllowed,
       SUM(CASE WHEN stat_name = 'takeaways'           THEN stat_value END) AS takeaways,
       SUM(CASE WHEN stat_name = 'completionsAttemptsAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '/') + 1) AS INTEGER) END) AS passAttempts,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, 1, INSTR(display_value, '-') - 1) AS INTEGER) END) AS thirdDownConv,
       SUM(CASE WHEN stat_name = 'thirdDownEffAllowed'
                THEN CAST(SUBSTR(display_value, INSTR(display_value, '-') + 1) AS INTEGER) END) AS thirdDownAtt
FROM Team_Defense_Opponent_Stats
GROUP BY game_id, team_id, opponent_id
""", conn)

game_player = pd.read_sql_query("""
SELECT game_id, team_id AS defense, stat_name, team_total
FROM Team_Defensive_Stats
WHERE stat_name IN ('sacks', 'tacklesForLoss', 'passesDefended')
""", conn).pivot_table(index=["game_id", "defense"], columns="stat_name",
                       values="team_total", aggfunc="sum").reset_index()

games_long = game_opp.merge(game_player, on=["game_id", "defense"], how="left")
games_long["offensePointsAllowed"] = games_long["offensePointsAllowed"].fillna(games_long["pointsAllowed"])
games_long[["sacks", "tacklesForLoss", "passesDefended"]] = (
    games_long[["sacks", "tacklesForLoss", "passesDefended"]].fillna(0))

with np.errstate(divide="ignore", invalid="ignore"):
    games_long["yardsPerPassAllowed"] = games_long["passingYards"] / games_long["passAttempts"]
    games_long["yardsPerRushAllowed"] = games_long["rushingYards"] / games_long["rushAttempts"]
    games_long["thirdDownRateAllowed"] = games_long["thirdDownConv"] / games_long["thirdDownAtt"]

# Pool FCS teams.
for side in ("defense", "offense"):
    games_long[side] = games_long[side].where(games_long[side].isin(fbs_ids), "FCS")

# Rate stats are weighted by attempts; everything else counts each game equally.
STAT_WEIGHTS = {
    "yardsPerPassAllowed": "passAttempts",
    "yardsPerRushAllowed": "rushAttempts",
    "thirdDownRateAllowed": "thirdDownAtt",
}

teams = sorted(set(games_long["defense"]) | set(games_long["offense"]))
team_pos = {t: i for i, t in enumerate(teams)}
n_teams = len(teams)

def fit_defense_strength(stat):
    """Ridge least squares fit of value = mu + off[offense] - def[defense]; returns def per team."""
    w_col = STAT_WEIGHTS.get(stat)
    df = games_long[np.isfinite(games_long[stat])]
    if w_col:
        df = df[df[w_col] > 0]
    w = df[w_col].to_numpy(float) if w_col else np.ones(len(df))
    w = w / w.mean()

    X = np.zeros((len(df), 1 + 2 * n_teams))
    rows = np.arange(len(df))
    X[:, 0] = 1.0
    X[rows, 1 + df["offense"].map(team_pos).to_numpy()] = 1.0
    X[rows, 1 + n_teams + df["defense"].map(team_pos).to_numpy()] = -1.0
    y = df[stat].to_numpy(float)

    # Weighted ridge: (X'WX + λI) b = X'Wy, without penalizing the intercept.
    penalty = RIDGE_PENALTY * np.eye(X.shape[1])
    penalty[0, 0] = 0.0
    XtW = X.T * w
    coef = np.linalg.solve(XtW @ X + penalty, XtW @ y)
    return pd.Series(coef[1 + n_teams:], index=teams)

# Defense strength per stat, in "better than average" units (positive = good defense
# for lower-is-better stats; flipped below for higher-is-better stats).
adj_strength = pd.DataFrame({stat: fit_defense_strength(stat) for stat in RANKING_STATS})
adj_strength = adj_strength.loc[adj_strength.index.isin(fbs_ids)]
for stat, (_, higher_is_better) in RANKING_STATS.items():
    if higher_is_better:
        adj_strength[stat] = -adj_strength[stat]   # model subtracts defense; flip so "more" is positive

# z-score across FBS teams and combine with the same weights as the raw ranking.
# (Signs are already "positive = better", so no second flip is needed.)
adj_z = (adj_strength - adj_strength.mean()) / adj_strength.std()
adjusted = fbs_games[["team", "conference", "games"]].join(adj_strength)
adjusted["adj_defense_score"] = (adj_z[stat_cols] * weights).sum(axis=1) / weights.sum()

# Bring in the raw rank so the effect of the adjustment is visible.
raw_rank = pd.Series(defense_rankings.index, index=defense_rankings["team_id"], name="raw_rank")
adjusted = adjusted.join(raw_rank)

adjusted_defense_rankings = (
    adjusted.sort_values("adj_defense_score", ascending=False)
    .reset_index()
    .rename_axis("rank")
)
adjusted_defense_rankings.index += 1
adjusted_defense_rankings["rank_change"] = adjusted_defense_rankings["raw_rank"] - adjusted_defense_rankings.index

display(Markdown(f"#### Opponent-adjusted defensive rankings, {YEAR} season — top 10"))
display(Markdown("Stat columns are per game (or per attempt) better than an average FBS defense "
                 "against the same schedule; positive is always better."))
display(adjusted_defense_rankings.head(10)[[
    "team", "conference", "adj_defense_score", "raw_rank", "rank_change",
    "offensePointsAllowed", "totalYardsAllowed", "yardsPerPassAllowed", "yardsPerRushAllowed",
    "thirdDownRateAllowed", "takeaways", "sacks",
]].round(2))

#### Opponent-adjusted defensive rankings, 2025 season — top 10

Stat columns are per game (or per attempt) better than an average FBS defense against the same schedule; positive is always better.

,team,conference,adj_defense_score,raw_rank,rank_change,offensePointsAllowed,totalYardsAllowed,yardsPerPassAllowed,yardsPerRushAllowed,thirdDownRateAllowed,takeaways,sacks
rank,,,,,,,,,,,,
1,Texas Tech Red Raiders,Big 12 Conference,2.14,1,0,11.81,106.05,1.54,1.98,0.08,0.97,1.03
2,Indiana Hoosiers,Big Ten Conference,1.94,3,1,14.10,106.11,1.27,1.14,0.10,0.66,1.02
3,Oklahoma Sooners,Southeastern Conference,1.81,2,-1,11.11,97.66,1.23,1.59,0.11,-0.09,1.17
4,Miami Hurricanes,Atlantic Coast Conference,1.67,7,3,13.40,101.04,1.59,1.42,0.11,0.24,1.09
5,Ohio State Buckeyes,Big Ten Conference,1.44,6,1,15.90,142.33,1.97,1.16,0.10,-0.01,0.50
6,Notre Dame Fighting Irish,FBS Independents,1.38,9,3,9.39,74.91,1.38,1.22,0.06,0.44,0.72
7,Texas A&M Aggies,Southeastern Conference,1.31,8,1,6.50,72.88,1.36,0.25,0.15,-0.37,0.93
8,Oregon Ducks,Big Ten Conference,1.27,15,7,9.09,99.73,1.85,0.84,0.05,0.24,0.08
9,Arizona Wildcats,Big 12 Conference,1.13,11,2,5.67,60.72,1.44,0.29,0.04,0.99,-0.21


CFP-committee-style weighting of the same stats: scoring defense (`offensePointsAllowed`) first, then efficiency and takeaways, with box-score volume stats weighted lightly. Applied to both the raw and the opponent-adjusted z-scores from the two cells above; the full table is stored in `cfp_defense_rankings`.

In [89]:
# CFP-committee-style weighting.
# The committee doesn't publish stat weights; it talks about results (scoring defense),
# strength of schedule, and the "eye test" of a dominant unit. So points allowed carries the
# most weight, efficiency stats (yards per play, 3rd down) and takeaways come next, and
# volume or box-score stats that the committee rarely cites (sacks, TFL, passes defended,
# first downs) get small weights. Weights are percentages and sum to 100.
CFP_WEIGHTS = {
    "offensePointsAllowed": 35,  # scoreboard result; the stat most often cited (offense-only points)
    "yardsPerPassAllowed":  12,  # efficiency, independent of pace or number of plays
    "yardsPerRushAllowed":  12,
    "takeaways":            12,  # changes games; shows up in highlights and broadcasts
    "totalYardsAllowed":    10,  # the familiar "total defense" number
    "thirdDownRateAllowed": 10,  # getting off the field
    "firstDownsAllowed":     3,  # mostly overlaps with yards allowed
    "sacks":                 3,  # how pressure appears on the stat sheet
    "tacklesForLoss":        2,
    "passesDefended":        1,
}
assert set(CFP_WEIGHTS) == set(RANKING_STATS) and sum(CFP_WEIGHTS.values()) == 100

cfp_w = pd.Series(CFP_WEIGHTS)[stat_cols]

# Same z-scores as the two rankings above (positive = better), just weighted differently.
raw_signed_z = (z * signs)[stat_cols]
cfp_scores = fbs_games[["team", "conference"]].copy()
cfp_scores["cfp_raw_score"] = raw_signed_z.mul(cfp_w).sum(axis=1) / cfp_w.sum()
cfp_scores["cfp_adj_score"] = adj_z[stat_cols].mul(cfp_w).sum(axis=1) / cfp_w.sum()
cfp_scores["cfp_raw_rank"] = cfp_scores["cfp_raw_score"].rank(ascending=False, method="min").astype(int)
cfp_scores = cfp_scores.join(
    adjusted_defense_rankings.reset_index().set_index("team_id")[["rank"]]
    .rename(columns={"rank": "equal_wt_adj_rank"}))

# The committee leans heavily on schedule strength, so the adjusted score is the headline ranking.
cfp_defense_rankings = (
    cfp_scores.sort_values("cfp_adj_score", ascending=False)
    .reset_index()
    .rename_axis("rank")
)
cfp_defense_rankings.index += 1

display(Markdown(f"#### CFP-weighted, opponent-adjusted defensive rankings, {YEAR} season — top 10"))
display(cfp_defense_rankings.head(10)[[
    "team", "conference", "cfp_adj_score", "equal_wt_adj_rank", "cfp_raw_rank",
]].round(2))

#### CFP-weighted, opponent-adjusted defensive rankings, 2025 season — top 10

,team,conference,cfp_adj_score,equal_wt_adj_rank,cfp_raw_rank
rank,,,,,
1,Texas Tech Red Raiders,Big 12 Conference,2.16,1,1
2,Indiana Hoosiers,Big Ten Conference,2.08,2,3
3,Ohio State Buckeyes,Big Ten Conference,2.04,5,2
4,Miami Hurricanes,Atlantic Coast Conference,1.92,4,6
5,Oklahoma Sooners,Southeastern Conference,1.71,3,5
6,Notre Dame Fighting Irish,FBS Independents,1.50,6,9
7,Oregon Ducks,Big Ten Conference,1.42,8,15
8,Iowa Hawkeyes,Big Ten Conference,1.34,15,14
9,Arizona Wildcats,Big 12 Conference,1.19,9,11


## 12. Quick look

In [90]:
pd.read_sql_query("""
SELECT g.week, g.game_date, h.display_name AS home, g.home_score,
       a.display_name AS away, g.away_score
FROM Games g
JOIN Teams h ON h.team_id = g.home_team_id
JOIN Teams a ON a.team_id = g.away_team_id
WHERE g.status = 'Final'
ORDER BY g.game_date DESC
LIMIT 10
""", conn)

,week,game_date,home,home_score,away,away_score
0,1,2026-01-20T00:30Z,Indiana Hoosiers,27,Miami Hurricanes,21
1,1,2026-01-10T00:30Z,Indiana Hoosiers,56,Oregon Ducks,22
2,1,2026-01-09T00:30Z,Ole Miss Rebels,27,Miami Hurricanes,31
3,1,2026-01-03T01:00Z,Mississippi State Bulldogs,29,Wake Forest Demon Deacons,43
4,1,2026-01-03T01:00Z,SMU Mustangs,24,Arizona Wildcats,19
5,1,2026-01-02T21:30Z,Cincinnati Bearcats,13,Navy Midshipmen,35
6,1,2026-01-02T18:00Z,Texas State Bobcats,41,Rice Owls,10
7,1,2026-01-02T01:00Z,Georgia Bulldogs,34,Ole Miss Rebels,39
8,1,2026-01-01T21:00Z,Indiana Hoosiers,38,Alabama Crimson Tide,3
9,1,2026-01-01T17:00Z,Texas Tech Red Raiders,0,Oregon Ducks,23


Every stat stored in the database (raw and derived), grouped by table and category, with how many rows each has and how many of those are numeric.

In [91]:
from IPython.display import Markdown, display

all_stats = pd.read_sql_query("""
SELECT 'Box_Scores' AS source, NULL AS stat_category, stat_name,
       MAX(stat_label) AS stat_label, COUNT(*) AS rows_, COUNT(stat_value) AS numeric_rows
FROM Box_Scores GROUP BY stat_name
UNION ALL
SELECT 'Player_Box_Scores', stat_category, stat_name,
       MAX(stat_label), COUNT(*), COUNT(stat_value)
FROM Player_Box_Scores GROUP BY stat_category, stat_name
UNION ALL
SELECT 'Team_Defensive_Stats', stat_category, stat_name,
       NULL, COUNT(*), COUNT(team_total)
FROM Team_Defensive_Stats GROUP BY stat_category, stat_name
UNION ALL
SELECT 'Team_Defense_Opponent_Stats', NULL, stat_name,
       MAX(stat_label), COUNT(*), COUNT(stat_value)
FROM Team_Defense_Opponent_Stats GROUP BY stat_name
ORDER BY source, stat_category, stat_name
""", conn)

all_stats["pct_numeric"] = (100 * all_stats["numeric_rows"] / all_stats["rows_"]).round(1).astype(str) + "%"
all_stats = all_stats.rename(columns={
    "stat_category": "Category", "stat_name": "Stat", "stat_label": "ESPN label",
    "rows_": "Rows", "numeric_rows": "Numeric rows", "pct_numeric": "% numeric",
})

with pd.option_context("display.max_rows", None):
    for source, group in all_stats.groupby("source", sort=False):
        table = group.drop(columns="source").dropna(axis="columns", how="all")
        index = ["Category", "Stat"] if "Category" in table else ["Stat"]
        display(Markdown(f"#### `{source}` ({len(table)} stats)"))
        display(table.set_index(index))

#### `Box_Scores` (15 stats)

,ESPN label,Rows,Numeric rows,% numeric
Stat,,,,
completionAttempts,Comp/Att,1916,0,0.0%
firstDowns,1st Downs,1916,1916,100.0%
fourthDownEff,4th down efficiency,1916,1608,83.9%
fumblesLost,Fumbles lost,1916,1916,100.0%
interceptions,Interceptions thrown,1916,1916,100.0%
netPassingYards,Passing,1916,1916,100.0%
possessionTime,Possession,1916,1916,100.0%
rushingAttempts,Rushing Attempts,1916,1916,100.0%
rushingYards,Rushing,1916,1916,100.0%


#### `Player_Box_Scores` (50 stats)

ESPN label   Rows  \
Category      Stat                                                   
defensive     defensiveTouchdowns                        TD  40277   
              hurries                                QB HUR  40277   
              passesDefended                             PD  40277   
              sacks                                   SACKS  40277   
              soloTackles                              SOLO  40277   
              tacklesForLoss                            TFL  40277   
              totalTackles                              TOT  40277   
fumbles       fumbles                                   FUM   2893   
              fumblesLost                              LOST   2893   
              fumblesRecovered                          REC   2893   
interceptions interceptionTouchdowns                     TD   1399   
              interceptionYards                         YDS   1399   
              interceptions                             INT   1399   
kickReturns   kickReturnTouchdowns                       TD   1959   
              kickReturnYards                           YDS   1959   
              kickReturns                                NO   1959   
              longKickReturn                           LONG   1959   
              yardsPerKickReturn                        AVG   1959   
kicking       extraPointsMade/extraPointAttempts         XP   2043   
              fieldGoalPct                              PCT   2043   
              fieldGoalsMade/fieldGoalAttempts           FG   2043   
              longFieldGoalMade                        LONG   2043   
              totalKickingPoints                        PTS   2043   
passing       adjQBR                                    QBR   3043   
              completions/passingAttempts             C/ATT   3123   
              interceptions                             INT   3123   
              passingTouchdowns                          TD   3123   
              passingYards                              YDS   3123   
              yardsPerPassAttempt                       AVG   3123   
puntReturns   longPuntReturn                           LONG   1472   
              puntReturnTouchdowns                       TD   1472   
              puntReturnYards                           YDS   1472   
              puntReturns                                NO   1472   
              yardsPerPuntReturn                        AVG   1472   
punting       grossAvgPuntYards                         AVG   2015   
              longPunt                                 LONG   2015   
              puntYards                                 YDS   2015   
              punts                                      NO   2015   
              puntsInside20                           In 20   2015   
              touchbacks                                 TB   2015   
receiving     longReception                            LONG  14236   
              receivingTouchdowns                        TD  14236   
              receivingYards                            YDS  14236   
              receptions                                REC  14236   
              yardsPerReception                         AVG  14236   
rushing       longRushing                              LONG  11060   
              rushingAttempts                           CAR  11060   
              rushingTouchdowns                          TD  11060   
              rushingYards                              YDS  11060   
              yardsPerRushAttempt                       AVG  11060   

                                                  Numeric rows % numeric  
Category      Stat                                                        
defensive     defensiveTouchdowns                        40277    100.0%  
              hurries                                    40277    100.0%  
              passesDefended                             40277    100.0%  
              sacks              

#### `Team_Defense_Opponent_Stats` (17 stats)

,ESPN label,Rows,Numeric rows,% numeric
Stat,,,,
completionsAttemptsAllowed,Comp/Att allowed,1916,0,0.0%
firstDownsAllowed,1st downs allowed,1916,1916,100.0%
fourthDownEffAllowed,4th down efficiency allowed,1916,1608,83.9%
fumblesRecovered,Fumbles recovered,1916,1916,100.0%
interceptionsMade,Interceptions made,1916,1916,100.0%
offensePointsAllowed,Points allowed to offense,1893,1893,100.0%
opponentPenaltiesYards,Opponent penalties,1916,0,0.0%
opponentPossessionTime,Opponent possession,1916,1916,100.0%
passingYardsAllowed,Passing yards allowed,1916,1916,100.0%


#### `Team_Defensive_Stats` (10 stats)

Rows  Numeric rows % numeric
Category      Stat                                                
defensive     defensiveTouchdowns     1870          1870    100.0%
              hurries                 1870          1870    100.0%
              passesDefended          1870          1870    100.0%
              sacks                   1870          1870    100.0%
              soloTackles             1870          1870    100.0%
              tacklesForLoss          1870          1870    100.0%
              totalTackles            1870          1870    100.0%
interceptions interceptionTouchdowns   999           999    100.0%
              interceptionYards        999           999    100.0%
              interceptions            999           999    100.0%

## Maintenance

- **Parsing change:** edit `parse_team_stats`, then run `rebuild_box_scores_from_raw()`. No requests needed.
- **Player parsing change:** edit `parse_player_stats`, then run `rebuild_player_box_scores_from_raw()` and `build_team_defensive_stats()`.
- **Play-by-play parsing change:** edit `parse_plays` / `parse_scoring_plays` (or `flatten_plays` in `map_schema.py`), then run `rebuild_plays_from_raw()` and `build_team_defense_opponent_stats()`. No requests needed.
- **Defensive aggregation change:** edit `build_team_defensive_stats` (or `DEFENSIVE_CATEGORIES`) and rerun it.
- **Opponent-based defense change:** edit `build_team_defense_opponent_stats` and rerun it; it only reads `Box_Scores`, `Games`, `Scoring_Plays` and `Raw_Plays`.
- **Force a team refresh:** `update_database(force_teams=True)`.
- **Remove the old derived table** from earlier experiments (derived stats are now computed on demand):

In [92]:
# conn.execute("DROP TABLE IF EXISTS Defense_Stats"); conn.commit()
# rebuild_box_scores_from_raw()
# rebuild_player_box_scores_from_raw()
# rebuild_plays_from_raw()
# build_team_defensive_stats()
# build_team_defense_opponent_stats()